# PALS: Efficient 17-Hypothesis Suite on OLMo-3 32B

**Optimized for A100 80GB + 235GB disk.** Key changes from v1:
- Pre-extracts all shared activations once (not per-experiment)
- Layer subsetting: 17 analysis layers instead of 64
- HF cache cleanup between model loads (saves ~130GB disk)
- H4 scans 8 targeted layers instead of 22 (saves ~30 min)
- Flash attention instead of eager (saves ~4GB VRAM)

**fp16 (no quantization) for clean steering vectors. Estimated runtime: ~60 min. Disk: ~80GB peak.**

In [ ]:
# === Cell 1: Setup ===
import subprocess, sys, os

if not os.path.exists('/content/clin-syco'):
    subprocess.run(['git', 'clone', 'https://github.com/elliott-leow/clin-syco.git', '/content/clin-syco'], check=True)

os.chdir('/content/clin-syco')
subprocess.run(['git', 'pull', '--ff-only'], check=False)  # always pull latest
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)

import torch
print(f'PyTorch {torch.__version__}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    print(f'Disk: {os.statvfs("/").f_bavail * os.statvfs("/").f_frsize / 1e9:.0f} GB free')
else:
    print('WARNING: No GPU detected')

In [ ]:
# === Cell 2: Configuration ===
import gc, json, os, time, platform, sys
from datetime import datetime, timezone
from glob import glob

import numpy as np
import torch
import torch.nn.functional as F
from IPython.display import display, Image

sys.path.insert(0, '/content/clin-syco/src')
sys.path.insert(0, '/content/clin-syco')

DEVICE = 'cuda'
STIMULI_DIR = '/content/clin-syco/stimuli'
OUTPUT_DIR = '/content/clin-syco/results/olmo3_32b_fp16'
os.makedirs(OUTPUT_DIR, exist_ok=True)

CHECKPOINTS = {
    'base': 'allenai/Olmo-3-1125-32B',
    'sft': 'allenai/Olmo-3.1-32B-Instruct-SFT',
    'dpo': 'allenai/Olmo-3.1-32B-Instruct-DPO',
}
DPO_MODEL_ID = CHECKPOINTS['dpo']

# Layer subsetting: 17 evenly-spaced layers for a 64-layer model
# Covers early, middle, late, and final layer
ANALYSIS_LAYERS = list(range(0, 64, 4)) + [63]  # [0,4,8,...,60,63] = 17 layers

# Stimuli counts for expensive experiments
N_STIMULI_H4 = 3
N_STIMULI_H12 = 10
N_STIMULI_H14 = 10
N_GENERATE_H16 = 3

# H4: only scan layers that showed signal in 1B pilot (layers 42-53 mapped from 1B's 12-15)
# For 64-layer model, last third = layers 42-63, but we target layers 42-53 (8 layers)
H4_TARGET_LAYERS = list(range(42, 54))

def disk_free_gb():
    s = os.statvfs('/')
    return s.f_bavail * s.f_frsize / 1e9

def vram_used_gb():
    return torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0

def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def clear_hf_cache(model_id=None):
    """Delete HuggingFace cache to free disk. Optionally only for a specific model."""
    import shutil
    cache_dir = os.path.expanduser('~/.cache/huggingface/hub')
    if model_id:
        # Convert model ID to cache directory name
        cache_name = 'models--' + model_id.replace('/', '--')
        path = os.path.join(cache_dir, cache_name)
        if os.path.exists(path):
            size = sum(os.path.getsize(os.path.join(dp, f)) for dp, dn, fn in os.walk(path) for f in fn)
            shutil.rmtree(path)
            print(f'  Cleared {size/1e9:.1f} GB from cache: {cache_name}')
    else:
        if os.path.exists(cache_dir):
            size = sum(os.path.getsize(os.path.join(dp, f)) for dp, dn, fn in os.walk(cache_dir) for f in fn)
            shutil.rmtree(cache_dir)
            os.makedirs(cache_dir)
            print(f'  Cleared entire HF cache: {size/1e9:.1f} GB')

def show_plots(output_dir):
    for p in sorted(glob(os.path.join(output_dir, '*.png'))):
        print(os.path.basename(p))
        display(Image(filename=p))

def build_metadata(model_id, device, model=None):
    meta = {'model': model_id, 'device': device, 'platform': platform.platform(), 'torch_version': torch.__version__}
    if model is not None:
        meta['num_layers'] = model.config.num_hidden_layers
        meta['hidden_size'] = model.config.hidden_size
        meta['num_params_M'] = round(sum(p.numel() for p in model.parameters()) / 1e6)
    return meta

def inject_metadata(result_json_path, metadata):
    with open(result_json_path) as f:
        data = json.load(f)
    data['metadata'] = metadata
    with open(result_json_path, 'w') as f:
        json.dump(data, f, indent=2)

print(f'Analysis layers: {len(ANALYSIS_LAYERS)} of 64')
print(f'H4 target layers: {len(H4_TARGET_LAYERS)}')
print(f'Disk free: {disk_free_gb():.0f} GB')

In [ ]:
# === Cell 3: Load DPO model (4-bit, flash attention) ===
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f'Loading {DPO_MODEL_ID} (fp16)...')
print(f'Disk free before: {disk_free_gb():.0f} GB')
t0 = time.time()

model = AutoModelForCausalLM.from_pretrained(
    DPO_MODEL_ID,
    torch_dtype=torch.float16,
    device_map='auto',
    attn_implementation='sdpa',
)
model.eval()
tokenizer = AutoTokenizer.from_pretrained(DPO_MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f'Loaded in {time.time()-t0:.0f}s')
print(f'Layers: {model.config.num_hidden_layers}, Hidden: {model.config.hidden_size}, Heads: {model.config.num_attention_heads}')
print(f'VRAM: {vram_used_gb():.1f} GB, Disk free: {disk_free_gb():.0f} GB')

In [ ]:
# === Cell 4: Pre-extract ALL shared activations ===
# This is the only expensive GPU phase. After this, most experiments are CPU-only.
from pals.extraction import batch_extract_contrastive
from pals.directions import compute_contrastive_direction, cosine_similarity_by_layer

# Load all stimuli
stimuli = {}
for name in ['cognitive_distortions', 'clinical_bridge', 'factual_control',
             'high_emotion_general', 'clinical_correct_answer', 'emotional_intensity_gradient']:
    with open(os.path.join(STIMULI_DIR, f'{name}.json')) as f:
        stimuli[name] = json.load(f)

L = ANALYSIS_LAYERS
print(f'Extracting activations at {len(L)} layers for all stimulus sets...\n')
t0 = time.time()

# Clinical sycophancy: sycophantic vs therapeutic
print('1/5 Clinical sycophancy...')
clin_pos, clin_neg = batch_extract_contrastive(
    model, tokenizer, stimuli['cognitive_distortions'],
    'sycophantic_completion', 'therapeutic_completion', layers=L, desc='Clinical')
clinical_dir = compute_contrastive_direction(clin_pos, clin_neg)
cleanup()

# Factual sycophancy
print('2/5 Factual sycophancy...')
fact_pos, fact_neg = batch_extract_contrastive(
    model, tokenizer, stimuli['factual_control'],
    'sycophantic_completion', 'therapeutic_completion', layers=L, desc='Factual')
factual_dir = compute_contrastive_direction(fact_pos, fact_neg)
cleanup()

# Bridge sycophancy
print('3/5 Bridge sycophancy...')
bridge_pos, bridge_neg = batch_extract_contrastive(
    model, tokenizer, stimuli['clinical_bridge'],
    'sycophantic_completion', 'therapeutic_completion', layers=L, desc='Bridge')
bridge_dir = compute_contrastive_direction(bridge_pos, bridge_neg)
cleanup()

# Empathy: therapeutic vs cold
print('4/5 Empathy...')
emp_pos, emp_neg = batch_extract_contrastive(
    model, tokenizer, stimuli['high_emotion_general'],
    'therapeutic_completion', 'cold_completion', layers=L, desc='Empathy')
empathy_dir = compute_contrastive_direction(emp_pos, emp_neg)
cleanup()

# Emotional intensity gradient
print('5/5 Emotional gradient...')
emo_grad = stimuli['emotional_intensity_gradient']
emo_by_level = {}
for level in [1, 2, 3]:
    items = [s for s in emo_grad if s.get('emotional_level') == level]
    if items:
        p, n = batch_extract_contrastive(
            model, tokenizer, items,
            'sycophantic_completion', 'therapeutic_completion', layers=L, desc=f'Emo-{level}')
        emo_by_level[level] = {'pos': p, 'neg': n, 'dir': compute_contrastive_direction(p, n)}
        cleanup()

elapsed = time.time() - t0
print(f'\nAll activations extracted in {elapsed:.0f}s ({elapsed/60:.1f} min)')
print(f'VRAM: {vram_used_gb():.1f} GB, Disk free: {disk_free_gb():.0f} GB')

In [ ]:
# === Cell 5: H1 — Direction Comparison (CPU-only) ===
from pals.probing import cross_domain_probing, within_domain_probing
from pals.stats import direction_similarity_permutation
from pals.viz import plot_cosine_similarity_by_layer, plot_probe_transfer

t0 = time.time()
h1_dir = os.path.join(OUTPUT_DIR, 'h1')
os.makedirs(h1_dir, exist_ok=True)

cos_clin_fact = cosine_similarity_by_layer(clinical_dir, factual_dir)
cos_bridge_fact = cosine_similarity_by_layer(bridge_dir, factual_dir)
cos_clin_bridge = cosine_similarity_by_layer(clinical_dir, bridge_dir)

all_layers = sorted(cos_clin_fact.keys())
print(f'H1 Cosine similarities (mean): clin-fact={np.mean(list(cos_clin_fact.values())):.3f}, '
      f'bridge-fact={np.mean(list(cos_bridge_fact.values())):.3f}, '
      f'clin-bridge={np.mean(list(cos_clin_bridge.values())):.3f}')

# Cross-domain probing
probe_f2c = cross_domain_probing(fact_pos, fact_neg, clin_pos, clin_neg, all_layers)
probe_c2f = cross_domain_probing(clin_pos, clin_neg, fact_pos, fact_neg, all_layers)
probe_within_c = within_domain_probing(clin_pos, clin_neg, all_layers)
probe_within_f = within_domain_probing(fact_pos, fact_neg, all_layers)

# Permutation tests at 3 layers
test_layers = [all_layers[len(all_layers)//4], all_layers[len(all_layers)//2], all_layers[3*len(all_layers)//4]]
perm_results = {}
for l in test_layers:
    res = direction_similarity_permutation(clin_pos, clin_neg, fact_pos, fact_neg, l, n_perms=2000)
    perm_results[l] = res
    print(f'  Perm test L{l}: cos={res["observed_cos"]:.3f}, p={res["p_value"]:.4f}')

results = {
    'cosine_similarities': {
        'clinical_vs_factual': {str(k): v for k, v in cos_clin_fact.items()},
        'bridge_vs_factual': {str(k): v for k, v in cos_bridge_fact.items()},
        'clinical_vs_bridge': {str(k): v for k, v in cos_clin_bridge.items()},
    },
    'cross_domain_probing': {
        'factual_to_clinical': {str(k): v for k, v in probe_f2c.items()},
        'clinical_to_factual': {str(k): v for k, v in probe_c2f.items()},
    },
    'within_domain_probing': {
        'clinical': {str(k): v for k, v in probe_within_c.items()},
        'factual': {str(k): v for k, v in probe_within_f.items()},
    },
    'permutation_tests': {str(l): {'observed_cos': r['observed_cos'], 'p_value': r['p_value']} for l, r in perm_results.items()},
}
with open(os.path.join(h1_dir, 'h1_results.json'), 'w') as f:
    json.dump(results, f, indent=2)

plot_cosine_similarity_by_layer(
    {'Clinical vs Factual': cos_clin_fact, 'Bridge vs Factual': cos_bridge_fact, 'Clinical vs Bridge': cos_clin_bridge},
    title='H1: Sycophancy Direction Similarity', save_path=os.path.join(h1_dir, 'h1_cosine_similarity.png'))

meta = build_metadata(DPO_MODEL_ID, DEVICE, model)
meta['timestamp'] = datetime.now(timezone.utc).isoformat()
meta['runtime_s'] = round(time.time()-t0, 1)
inject_metadata(os.path.join(h1_dir, 'h1_results.json'), meta)
print(f'H1 done in {time.time()-t0:.0f}s')
show_plots(h1_dir)

In [ ]:
# === Cell 6: H2 — Uncertainty vs Deference (needs GPU for logit lens) ===
from experiments.h2_uncertainty_deference import run as run_h2

t0 = time.time()
h2_dir = os.path.join(OUTPUT_DIR, 'h2')
run_h2(model, tokenizer, STIMULI_DIR, h2_dir, layers=ANALYSIS_LAYERS)
meta = build_metadata(DPO_MODEL_ID, DEVICE, model)
meta['timestamp'] = datetime.now(timezone.utc).isoformat()
meta['runtime_s'] = round(time.time()-t0, 1)
inject_metadata(os.path.join(h2_dir, 'h2_results.json'), meta)
cleanup()
print(f'H2 done in {time.time()-t0:.0f}s')
show_plots(h2_dir)

In [ ]:
# === Cell 7: H6, H7, H9, H10, H11, H13 — CPU-only experiments using pre-extracted data ===
# These all just need the pre-computed directions and activations

from pals.decomposition import decompose_by_layer, decompose_ols_by_layer

# --- H6: Emotional Gradient ---
t0 = time.time()
h6_dir = os.path.join(OUTPUT_DIR, 'h6')
os.makedirs(h6_dir, exist_ok=True)

cos_by_level = {}
for level, data in sorted(emo_by_level.items()):
    cos_by_level[level] = cosine_similarity_by_layer(data['dir'], factual_dir)

mean_by_level = {l: np.mean(list(c.values())) for l, c in cos_by_level.items()}
h6_results = {
    'cosine_by_level_and_layer': {str(l): {str(k): v for k, v in c.items()} for l, c in cos_by_level.items()},
    'mean_cosine_by_level': {str(l): v for l, v in mean_by_level.items()},
    'trend_monotonic_decrease': len(mean_by_level) >= 2 and all(mean_by_level.get(i, 0) > mean_by_level.get(i+1, 1) for i in range(1, max(mean_by_level.keys()))),
}
with open(os.path.join(h6_dir, 'h6_results.json'), 'w') as f:
    json.dump(h6_results, f, indent=2)
print(f'H6 done in {time.time()-t0:.0f}s — levels: {mean_by_level}')

# --- H9: Warmth Tax ---
t0 = time.time()
h9_dir = os.path.join(OUTPUT_DIR, 'h9')
os.makedirs(h9_dir, exist_ok=True)

decomp = decompose_by_layer(clinical_dir, {'empathy': empathy_dir, 'factual_sycophancy': factual_dir})
h9_layers = sorted(decomp.keys())
mean_emp = np.mean([decomp[l]['unique_variance_explained']['empathy'] for l in h9_layers])
mean_fact = np.mean([decomp[l]['unique_variance_explained']['factual_sycophancy'] for l in h9_layers])
mean_res = np.mean([decomp[l]['residual_variance_fraction'] for l in h9_layers])

h9_results = {
    'mean_unique_variance_explained': {'empathy': float(mean_emp), 'factual_sycophancy': float(mean_fact), 'residual': float(mean_res)},
    'decomposition_by_layer': {str(l): {
        'empathy_unique_ve': decomp[l]['unique_variance_explained']['empathy'],
        'factual_unique_ve': decomp[l]['unique_variance_explained']['factual_sycophancy'],
        'residual_variance_fraction': decomp[l]['residual_variance_fraction'],
    } for l in h9_layers}
}
with open(os.path.join(h9_dir, 'h9_results.json'), 'w') as f:
    json.dump(h9_results, f, indent=2)
print(f'H9 done in {time.time()-t0:.0f}s — empathy={mean_emp:.1%}, factual={mean_fact:.1%}, residual={mean_res:.1%}')

# --- H10: Extended Decomposition ---
t0 = time.time()
h10_dir = os.path.join(OUTPUT_DIR, 'h10')
from experiments.h10_extended_decomposition import run as run_h10
run_h10(model, tokenizer, STIMULI_DIR, h10_dir, layers=ANALYSIS_LAYERS)
cleanup()
print(f'H10 done in {time.time()-t0:.0f}s')

# --- H11: Orthogonal Complement ---
t0 = time.time()
h11_dir = os.path.join(OUTPUT_DIR, 'h11')
from experiments.h11_orthogonal_complement import run as run_h11
run_h11(model, tokenizer, STIMULI_DIR, h11_dir, layers=ANALYSIS_LAYERS)
cleanup()
print(f'H11 done in {time.time()-t0:.0f}s')

# --- H7: Distortion Subspaces ---
t0 = time.time()
h7_dir = os.path.join(OUTPUT_DIR, 'h7')
from experiments.h7_distortion_subspaces import run as run_h7
run_h7(model, tokenizer, STIMULI_DIR, h7_dir, layers=ANALYSIS_LAYERS)
cleanup()
print(f'H7 done in {time.time()-t0:.0f}s')

# --- H13: Distortion x Emotion Interaction ---
t0 = time.time()
h13_dir = os.path.join(OUTPUT_DIR, 'h13')
from experiments.h13_distortion_emotion_interaction import run as run_h13
run_h13(model, tokenizer, STIMULI_DIR, h13_dir, layers=ANALYSIS_LAYERS)
cleanup()
print(f'H13 done in {time.time()-t0:.0f}s')

# Inject metadata for CPU experiments
for hdir in [h6_dir, h9_dir]:
    for jf in glob(os.path.join(hdir, '*_results.json')):
        inject_metadata(jf, {**build_metadata(DPO_MODEL_ID, DEVICE, model),
                             'timestamp': datetime.now(timezone.utc).isoformat(),
                             'analysis_layers': ANALYSIS_LAYERS})

print(f'\nAll CPU experiments done. VRAM: {vram_used_gb():.1f} GB, Disk free: {disk_free_gb():.0f} GB')

In [ ]:
# === Cell 8: H4 — Attention Routing (GPU, targeted layers) ===
from pals.attention import find_sycophancy_routing_heads

t0 = time.time()
h4_dir = os.path.join(OUTPUT_DIR, 'h4')
os.makedirs(h4_dir, exist_ok=True)

clinical_sub = stimuli['cognitive_distortions'][:N_STIMULI_H4]
factual_sub = stimuli['factual_control'][:N_STIMULI_H4]
n_heads = model.config.num_attention_heads

print(f'H4: Scanning {len(H4_TARGET_LAYERS)} layers x {n_heads} heads x {N_STIMULI_H4} stimuli')
print(f'  ({len(H4_TARGET_LAYERS) * n_heads * N_STIMULI_H4 * 2} forward passes)')

clinical_effects = find_sycophancy_routing_heads(model, tokenizer, clinical_sub, H4_TARGET_LAYERS, n_heads)
cleanup()
factual_effects = find_sycophancy_routing_heads(model, tokenizer, factual_sub, H4_TARGET_LAYERS, n_heads)
cleanup()

clin_sorted = sorted(clinical_effects.items(), key=lambda x: x[1])
fact_sorted = sorted(factual_effects.items(), key=lambda x: x[1])
top_k = 10
clin_top = set(k for k, v in clin_sorted[:top_k])
fact_top = set(k for k, v in fact_sorted[:top_k])
overlap = clin_top & fact_top

h4_results = {
    'clinical_head_effects': {f'L{l}H{h}': v for (l,h), v in clinical_effects.items()},
    'factual_head_effects': {f'L{l}H{h}': v for (l,h), v in factual_effects.items()},
    'clinical_top10': [{'layer': l, 'head': h, 'shift': s} for (l,h), s in clin_sorted[:10]],
    'factual_top10': [{'layer': l, 'head': h, 'shift': s} for (l,h), s in fact_sorted[:10]],
    'overlap_count': len(overlap), 'overlap_heads': [list(x) for x in overlap],
}
with open(os.path.join(h4_dir, 'h4_results.json'), 'w') as f:
    json.dump(h4_results, f, indent=2)

inject_metadata(os.path.join(h4_dir, 'h4_results.json'),
    {**build_metadata(DPO_MODEL_ID, DEVICE, model), 'timestamp': datetime.now(timezone.utc).isoformat(),
     'runtime_s': round(time.time()-t0, 1), 'target_layers': H4_TARGET_LAYERS})
print(f'H4 done in {time.time()-t0:.0f}s — overlap: {len(overlap)}/{top_k}')

In [ ]:
# === Cell 9: H12, H14, H15, H16, H17 — GPU experiments ===

# H12: Activation Addition
t0 = time.time()
from experiments.h12_activation_addition import run as run_h12
h12_dir = os.path.join(OUTPUT_DIR, 'h12')
run_h12(model, tokenizer, STIMULI_DIR, h12_dir, layers=ANALYSIS_LAYERS, n_stimuli=N_STIMULI_H12)
cleanup()
inject_metadata(os.path.join(h12_dir, 'h12_results.json'),
    {**build_metadata(DPO_MODEL_ID, DEVICE, model), 'timestamp': datetime.now(timezone.utc).isoformat(), 'runtime_s': round(time.time()-t0, 1)})
print(f'H12 done in {time.time()-t0:.0f}s')

# H14: Token Attribution
t0 = time.time()
from experiments.h14_token_attribution import run as run_h14
h14_dir = os.path.join(OUTPUT_DIR, 'h14')
run_h14(model, tokenizer, STIMULI_DIR, h14_dir, layers=ANALYSIS_LAYERS, n_stimuli=N_STIMULI_H14)
cleanup()
inject_metadata(os.path.join(h14_dir, 'h14_results.json'),
    {**build_metadata(DPO_MODEL_ID, DEVICE, model), 'timestamp': datetime.now(timezone.utc).isoformat(), 'runtime_s': round(time.time()-t0, 1)})
print(f'H14 done in {time.time()-t0:.0f}s')

# H15: Circuit Decomposition
t0 = time.time()
from experiments.h15_circuit_decomposition import run as run_h15
h15_dir = os.path.join(OUTPUT_DIR, 'h15')
run_h15(model, tokenizer, STIMULI_DIR, h15_dir, layers=ANALYSIS_LAYERS)
cleanup()
inject_metadata(os.path.join(h15_dir, 'h15_results.json'),
    {**build_metadata(DPO_MODEL_ID, DEVICE, model), 'timestamp': datetime.now(timezone.utc).isoformat(), 'runtime_s': round(time.time()-t0, 1)})
print(f'H15 done in {time.time()-t0:.0f}s')

# H16: Publication Validation
t0 = time.time()
from experiments.h16_publication_validation import run as run_h16
h16_dir = os.path.join(OUTPUT_DIR, 'h16')
run_h16(model, tokenizer, STIMULI_DIR, h16_dir, layers=ANALYSIS_LAYERS, n_generate=N_GENERATE_H16, max_new_tokens=30)
cleanup()
inject_metadata(os.path.join(h16_dir, 'h16_results.json'),
    {**build_metadata(DPO_MODEL_ID, DEVICE, model), 'timestamp': datetime.now(timezone.utc).isoformat(), 'runtime_s': round(time.time()-t0, 1)})
print(f'H16 done in {time.time()-t0:.0f}s')

# H17: Methodology Validation
t0 = time.time()
from experiments.h17_methodology_validation import run as run_h17
h17_dir = os.path.join(OUTPUT_DIR, 'h17')
run_h17(model, tokenizer, STIMULI_DIR, h17_dir, layers=ANALYSIS_LAYERS)
cleanup()
inject_metadata(os.path.join(h17_dir, 'h17_results.json'),
    {**build_metadata(DPO_MODEL_ID, DEVICE, model), 'timestamp': datetime.now(timezone.utc).isoformat(), 'runtime_s': round(time.time()-t0, 1)})
print(f'H17 done in {time.time()-t0:.0f}s')

print(f'\nAll Phase 1 done. VRAM: {vram_used_gb():.1f} GB, Disk free: {disk_free_gb():.0f} GB')

In [ ]:
# === Cell 10: Free DPO model + clear HF cache before Phase 2 ===
print(f'Before cleanup: VRAM={vram_used_gb():.1f} GB, Disk free={disk_free_gb():.0f} GB')

# Free pre-extracted activations (no longer needed)
del clin_pos, clin_neg, fact_pos, fact_neg, bridge_pos, bridge_neg, emp_pos, emp_neg
del clinical_dir, factual_dir, bridge_dir, empathy_dir, emo_by_level
del model, tokenizer
cleanup()

# Clear the DPO model from HF cache to free ~64GB disk
clear_hf_cache(DPO_MODEL_ID)

print(f'After cleanup: VRAM={vram_used_gb():.1f} GB, Disk free={disk_free_gb():.0f} GB')
print('Ready for Phase 2 (multi-checkpoint)')

In [ ]:
# === Cell 11: Phase 2 — H3, H5, H8 (multi-checkpoint, sequential load) ===
# Each experiment loads checkpoints internally. We clear cache between them.

# H3: Checkpoint Evolution
t0 = time.time()
from experiments.h3_checkpoint_evolution import run as run_h3
h3_dir = os.path.join(OUTPUT_DIR, 'h3')
run_h3(None, None, STIMULI_DIR, h3_dir,
       checkpoints=CHECKPOINTS, device=DEVICE, quantize_4bit=False, layers=ANALYSIS_LAYERS)
cleanup()
inject_metadata(os.path.join(h3_dir, 'h3_results.json'),
    {'model': 'multi-checkpoint', 'device': DEVICE, 'timestamp': datetime.now(timezone.utc).isoformat(),
     'runtime_s': round(time.time()-t0, 1), 'checkpoints': CHECKPOINTS})
print(f'H3 done in {time.time()-t0:.0f}s. Disk free: {disk_free_gb():.0f} GB')

# Clear all cached models before next experiment
clear_hf_cache()
print(f'After cache clear: Disk free={disk_free_gb():.0f} GB')

# H5: DPO Reversal
t0 = time.time()
from experiments.h5_dpo_reversal import run as run_h5
h5_dir = os.path.join(OUTPUT_DIR, 'h5')
run_h5(None, None, STIMULI_DIR, h5_dir,
       checkpoints=CHECKPOINTS, device=DEVICE, quantize_4bit=False)
cleanup()
inject_metadata(os.path.join(h5_dir, 'h5_results.json'),
    {'model': 'multi-checkpoint', 'device': DEVICE, 'timestamp': datetime.now(timezone.utc).isoformat(),
     'runtime_s': round(time.time()-t0, 1), 'checkpoints': CHECKPOINTS})
print(f'H5 done in {time.time()-t0:.0f}s. Disk free: {disk_free_gb():.0f} GB')

clear_hf_cache()

# H8: Pretraining Entanglement
t0 = time.time()
from experiments.h8_pretraining_entanglement import run as run_h8
h8_dir = os.path.join(OUTPUT_DIR, 'h8')
run_h8(None, None, STIMULI_DIR, h8_dir,
       checkpoints=CHECKPOINTS, device=DEVICE, quantize_4bit=False, layers=ANALYSIS_LAYERS)
cleanup()
inject_metadata(os.path.join(h8_dir, 'h8_results.json'),
    {'model': 'multi-checkpoint', 'device': DEVICE, 'timestamp': datetime.now(timezone.utc).isoformat(),
     'runtime_s': round(time.time()-t0, 1), 'checkpoints': CHECKPOINTS})
print(f'H8 done in {time.time()-t0:.0f}s. Disk free: {disk_free_gb():.0f} GB')

clear_hf_cache()

In [ ]:
# === Cell 12: Summary ===
hypotheses = [
    ('H1','Direction Comparison','h1/h1_results.json'),
    ('H2','Uncertainty vs Deference','h2/h2_results.json'),
    ('H3','Checkpoint Evolution','h3/h3_results.json'),
    ('H4','Attention Routing','h4/h4_results.json'),
    ('H5','DPO Reversal','h5/h5_results.json'),
    ('H6','Emotional Gradient','h6/h6_results.json'),
    ('H7','Distortion Subspaces','h7/h7_results.json'),
    ('H8','Pretraining Entanglement','h8/h8_results.json'),
    ('H9','Warmth Tax','h9/h9_results.json'),
    ('H10','Extended Decomposition','h10/h10_results.json'),
    ('H11','Orthogonal Complement','h11/h11_results.json'),
    ('H12','Activation Addition','h12/h12_results.json'),
    ('H13','Distortion x Emotion','h13/h13_results.json'),
    ('H14','Token Attribution','h14/h14_results.json'),
    ('H15','Circuit Decomposition','h15/h15_results.json'),
    ('H16','Publication Validation','h16/h16_results.json'),
    ('H17','Methodology Validation','h17/h17_results.json'),
]
print('=' * 80)
print(f'{"Hyp":<5} {"Name":<30} {"Runtime":>10} {"Status"}')
print('=' * 80)
total = 0
for hid, name, rpath in hypotheses:
    fp = os.path.join(OUTPUT_DIR, rpath)
    if os.path.exists(fp):
        with open(fp) as f:
            d = json.load(f)
        rt = d.get('metadata', {}).get('runtime_s', '?')
        if isinstance(rt, (int, float)): total += rt
        print(f'{hid:<5} {name:<30} {str(rt)+"s":>10} OK')
    else:
        print(f'{hid:<5} {name:<30} {"":>10} MISSING')
print('=' * 80)
print(f'Total: {total:.0f}s ({total/60:.1f} min)')

In [ ]:
# === Cell 13: Download results ===
import shutil
zip_path = '/content/pals_olmo3_32b_fp16_results'
shutil.make_archive(zip_path, 'zip', OUTPUT_DIR)
print(f'Archive: {zip_path}.zip ({os.path.getsize(zip_path + ".zip") / 1e6:.1f} MB)')
try:
    from google.colab import files
    files.download(zip_path + '.zip')
except ImportError:
    print(f'Download manually: {zip_path}.zip')